# 00e. TensorFlow 2 & Keras Core Mechanisms (80/20 Mastery Guide)

Many biomedical & bioinformatics repos (including **PaccMann** and several TCGA/drug-response papers) are written in **TensorFlow 2 / Keras** instead of PyTorch.

Good news: **TensorFlow 2 works cleanly at TWO levels** you already understand:
1. **High-level Keras API (`Sequential`, `Functional`, `compile`/`fit`)** — builds and trains standard models in 5 lines.
2. **Low-level Research API (`tf.Variable`, `tf.GradientTape`, `keras.Model` subclassing)** — looks almost identical to PyTorch's `nn.Module` + training loop!

### What You Will Master (80/20 Rule):
1. `tf.Tensor` (immutable) vs `tf.Variable` (trainable weights!)
2. Autograd with **`tf.GradientTape()`**
3. Three ways to build Keras models: **Sequential**, **Functional** (multi-input!), and **Subclassing (`tf.keras.Model`)**
4. `tf.data.Dataset` pipeline (TF's `DataLoader`)
5. Fast `model.compile()` + `model.fit()` vs. Custom `GradientTape` training loop
6. Side-by-side **PyTorch ↔ TensorFlow cheat sheet**

## 1. `tf.Tensor` (Fixed Data) vs. `tf.Variable` (Trainable Weights)

Unlike PyTorch (where a `torch.Tensor` with `requires_grad=True` acts as both), TensorFlow separates them clearly:
* **`tf.constant(...)` / `tf.Tensor`**: **Immutable** (cannot be modified in-place; used for inputs, outputs, constants).
* **`tf.Variable(...)`**: **Mutable** (trainable weights updated by an optimizer!).
* Use `.numpy()` to convert back to a NumPy array anytime.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide noisy TF startup logs

import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

x = tf.constant([[1.0, 2.0], [3.0, 4.0]], dtype=tf.float32)  # fixed tensor
w = tf.Variable([[0.5], [-0.5]], dtype=tf.float32)           # trainable weight

y = x @ w  # matrix multiply works just like NumPy!
print("Result Tensor:\n", y)
print("As NumPy array:\n", y.numpy())

## 2. Automatic Differentiation: `tf.GradientTape()`

Remember PyTorch's `loss.backward()`?  
TensorFlow uses a **`with tf.GradientTape() as tape:`** block instead!
* Think of `tape` like a **video recorder**: every calculation involving a `tf.Variable` inside the `with` block is recorded onto the tape.
* Afterward, calling **`tape.gradient(loss, variables)`** rewinds the tape and computes exact derivatives!
* *(By default, TF automatically watches all `tf.Variable`s; for plain `tf.Tensor`s, call `tape.watch(t)`.)*

In [ ]:
# Same exact calculus check as PyTorch notebook:
# f(w) = (w * x - y)^2 at w=3, x=2, y=10  ->  df/dw = -16.0
w = tf.Variable(3.0)
x = tf.constant(2.0)
y = tf.constant(10.0)

with tf.GradientTape() as tape:
    loss = (w * x - y) ** 2

grad_w = tape.gradient(loss, w)  # Computes df/dw
print("Loss:", loss.numpy(), "| df/dw:", grad_w.numpy())

## 3. Building Models — Way 1: Keras `Sequential`

Just like `nn.Sequential` in PyTorch, stack layers one after another in a list!
Notice one HUGE convenience in Keras: **you do NOT have to compute `in_features` manually for `Dense` layers** — Keras infers the input shape automatically the first time data passes through!

> **Shape Difference vs. PyTorch:**  
> * PyTorch `Conv2d` uses **channels-first**: `(Batch, Channels, H, W)`  
> * TensorFlow `Conv2D` defaults to **channels-last**: `(Batch, H, W, Channels)`!

In [ ]:
from tensorflow.keras import layers

seq_model = tf.keras.Sequential([
    layers.Input(shape=(8,)),             # 8 input features per sample
    layers.Dense(32, activation="relu"),  # no need to manually specify in_dim!
    layers.Dropout(0.2),
    layers.Dense(2),                      # 2 raw logits (no softmax!)
])

seq_model.summary()
dummy_x = tf.random.normal((4, 8))
print("Sequential output shape:", seq_model(dummy_x).shape)  # (4, 2)

## 4. Building Models — Way 2: Keras **Functional API** (Multi-Input & Skip Connections)

In drug response (e.g., **PaccMann**: drug SMILES + tumor gene expression) or ResNets (skip connections), a single linear stack (`Sequential`) is NOT enough!

The **Functional API** lets you wire layers like LEGO blocks by calling each layer like a function `h = Layer(...)(prev_tensor)` and combining branches with `layers.Concatenate()` or `layers.Add()`.

In [ ]:
# Multimodal example: Drug SMILES tokens + Gene Expression vector -> IC50 prediction!
smiles_in = layers.Input(shape=(12,), dtype="int32", name="smiles_tokens")
omics_in = layers.Input(shape=(20,), dtype="float32", name="gene_expr")

# Branch 1: Sequence encoder for SMILES
x1 = layers.Embedding(input_dim=30, output_dim=16)(smiles_in)
x1 = layers.GRU(32)(x1)                      # outputs final state (Batch, 32)

# Branch 2: Dense encoder for Gene Expression
x2 = layers.Dense(32, activation="relu")(omics_in)

# Fuse both branches -> predict scalar IC50
fused = layers.Concatenate()([x1, x2])       # (Batch, 64)
out = layers.Dense(1, name="ic50_pred")(fused)

multimodal_model = tf.keras.Model(inputs=[smiles_in, omics_in], outputs=out)

fake_smiles = tf.random.uniform((5, 12), maxval=30, dtype=tf.int32)
fake_omics = tf.random.normal((5, 20))
print("Multimodal IC50 output shape:", multimodal_model([fake_smiles, fake_omics]).shape)

## 5. Building Models — Way 3: Subclassing `tf.keras.Model` (PyTorch Twin!)

If you love PyTorch's `nn.Module`, Keras **Model Subclassing** is nearly line-for-line identical—with two name swaps:
1. Inherit from `tf.keras.Model` (call `super().__init__()`).
2. Name the method **`call(self, inputs, training=False)`** instead of `forward(self, x)`!
   * `training=False` tells `Dropout` / `BatchNormalization` whether we are training or evaluating!

In [ ]:
class SubclassedMLP(tf.keras.Model):
    def __init__(self, hidden_dim: int, num_classes: int):
        super().__init__()  # MANDATORY just like PyTorch!
        self.fc1 = layers.Dense(hidden_dim, activation="relu")
        self.drop = layers.Dropout(0.2)
        self.head = layers.Dense(num_classes)

    def call(self, x, training=False):
        h = self.fc1(x)
        h = self.drop(h, training=training)  # Dropout ONLY active when training=True!
        return self.head(h)

sub_model = SubclassedMLP(hidden_dim=32, num_classes=2)
out = sub_model(tf.random.normal((4, 8)), training=False)
print("Subclassed output shape:", out.shape)
print("Trainable weight tensors:", len(sub_model.trainable_variables))

## 6. `tf.data.Dataset`: TensorFlow's `DataLoader`

Instead of PyTorch's `Dataset + DataLoader`, TensorFlow chains pipeline methods cleanly onto `tf.data.Dataset`:
* `.from_tensor_slices((X, y))` — wraps NumPy arrays or tensors
* `.shuffle(buffer_size)` — shuffles samples
* `.batch(32)` — groups into mini-batches
* `.prefetch(tf.data.AUTOTUNE)` — loads the next batch on CPU while GPU trains!

In [ ]:
np.random.seed(42)
X_np = np.random.randn(200, 8).astype(np.float32)
y_np = ((X_np[:, 0] ** 2 + X_np[:, 1] ** 2) > 1.5).astype(np.int32)

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_np[:160], y_np[:160]))
    .shuffle(buffer_size=160)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((X_np[160:], y_np[160:])).batch(32)

xb, yb = next(iter(train_ds))
print("Batch shapes:", xb.shape, yb.shape)

## 7. Fast Training: `model.compile()` + `model.fit()`

For standard supervised learning, you don't even have to write a loop in Keras:
* **`from_logits=True`** in `SparseCategoricalCrossentropy(from_logits=True)` is TensorFlow's equivalent of PyTorch's `nn.CrossEntropyLoss()` (it takes raw logits + integer class IDs `0, 1, 2`!).

In [ ]:
keras_clf = SubclassedMLP(hidden_dim=32, num_classes=2)

keras_clf.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

history = keras_clf.fit(train_ds, validation_data=val_ds, epochs=15, verbose=0)
print("Final train acc:", round(history.history["accuracy"][-1], 4))
print("Final val acc:  ", round(history.history["val_accuracy"][-1], 4))

## 8. Research Training: Custom `GradientTape` Loop (From-Scratch Control)

When reproducing custom research losses (like **DDANN domain adversarial loss** or **Cox partial log-likelihood**), `.fit()` can feel restrictive. Here is how to write the exact PyTorch-style loop in TensorFlow:
1. Inside `with tf.GradientTape() as tape:` → run `logits = model(xb, training=True)` and `loss = loss_fn(yb, logits)`.
2. `grads = tape.gradient(loss, model.trainable_variables)`
3. `optimizer.apply_gradients(zip(grads, model.trainable_variables))`

In [ ]:
custom_model = SubclassedMLP(hidden_dim=32, num_classes=2)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt = tf.keras.optimizers.Adam(learning_rate=1e-2)

for epoch in range(1, 16):
    total_loss = 0.0
    steps = 0
    for xb, yb in train_ds:
        with tf.GradientTape() as tape:
            logits = custom_model(xb, training=True)
            loss = loss_fn(yb, logits)
        grads = tape.gradient(loss, custom_model.trainable_variables)
        opt.apply_gradients(zip(grads, custom_model.trainable_variables))
        total_loss += float(loss)
        steps += 1

    if epoch in (1, 5, 10, 15):
        # Quick validation check
        val_logits = custom_model(X_np[160:], training=False)
        val_preds = tf.argmax(val_logits, axis=1, output_type=tf.int32).numpy()
        val_acc = np.mean(val_preds == y_np[160:])
        print(f"Epoch {epoch:2d} | train_loss={total_loss/steps:.4f} | val_acc={val_acc:.2%}")

## 9. PyTorch ↔ TensorFlow 80/20 Translation Cheat Sheet

| Concept | PyTorch | TensorFlow 2 / Keras |
|---|---|---|
| Fixed data tensor | `torch.tensor(...)` | `tf.constant(...)` |
| Trainable weights | `torch.tensor(..., requires_grad=True)` / `nn.Parameter` | `tf.Variable(...)` |
| Model base class | `class Net(nn.Module):` + `forward(self, x)` | `class Net(tf.keras.Model):` + `call(self, x, training=False)` |
| Conv2D image layout | `(B, C, H, W)` (channels first) | `(B, H, W, C)` (channels last) |
| RNN / LSTM / GRU | `nn.LSTM(..., batch_first=True)` | `layers.LSTM(...)` *(already batch-first!)* |
| Autograd | `loss.backward()` | `with tf.GradientTape() as tape:` + `tape.gradient(...)` |
| Weight update | `optimizer.step()` | `optimizer.apply_gradients(zip(grads, vars))` |
| Train vs Eval mode | `model.train()` / `model.eval()` | `training=True` / `training=False` |
| Logits loss (int labels) | `nn.CrossEntropyLoss()` | `SparseCategoricalCrossentropy(from_logits=True)` |
| Data pipeline | `Dataset` + `DataLoader` | `tf.data.Dataset.from_tensor_slices(...)` |

---
## 🎯 Practice Challenges (Try Without Peeking!)

1. Modify `SubclassedMLP` to accept a **2nd input** inside `call(self, x, training=False)` or add a **Residual skip connection** (`h = h + x` when dimensions match!).
2. Convert the `multimodal_model` in Section 4 to predict **2 classes** instead of 1 scalar, compile it with `SparseCategoricalCrossentropy(from_logits=True)`, and run 3 epochs of `.fit()` on `(fake_smiles, fake_omics)`.